<a href="https://colab.research.google.com/github/Jamieho1602/IBM-HR-Analytics-Overview/blob/main/IBM_HR_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# Load the five IBM HR Analytics datasets from GitHub
import pandas as pd

employees = pd.read_csv('https://raw.githubusercontent.com/Jamieho1602/IBM-HR-Analytics-Overview/refs/heads/main/Employees.csv')

compensation = pd.read_csv('https://raw.githubusercontent.com/Jamieho1602/IBM-HR-Analytics-Overview/refs/heads/main/Compensation.csv')

performance = pd.read_csv('https://raw.githubusercontent.com/Jamieho1602/IBM-HR-Analytics-Overview/refs/heads/main/Performance.csv')

experience = pd.read_csv('https://raw.githubusercontent.com/Jamieho1602/IBM-HR-Analytics-Overview/refs/heads/main/Experience.csv')

employment_status = pd.read_csv('https://raw.githubusercontent.com/Jamieho1602/IBM-HR-Analytics-Overview/refs/heads/main/Employment%20Status.csv')

# Check the number of rows and columns
print(employees.shape)
print(compensation.shape)
print(performance.shape)
print(experience.shape)
print(employment_status.shape)

(1470, 14)
(1470, 7)
(1470, 7)
(1470, 8)
(1470, 3)


In [31]:
##Check for missing values.
print("Employees Missing Values")
print(employees.isnull().sum())

print("Compensation Missing Values")
print(compensation.isnull().sum())

print("Performance Missing Values")
print(performance.isnull().sum())

print("Experience Missing Values")
print(experience.isnull().sum())

print("Employment Status Missing Values")
print(employment_status.isnull().sum())


Employees Missing Values
EmployeeNumber      0
Age                 5
Gender              0
MaritalStatus       0
Education           0
EducationField      0
Department          0
JobRole             0
JobLevel            0
BusinessTravel      0
DistanceFromHome    0
EmployeeCount       0
Over18              0
StandardHours       0
dtype: int64
Compensation Missing Values
EmployeeNumber       0
DailyRate            0
HourlyRate           0
MonthlyIncome        0
MonthlyRate          0
PercentSalaryHike    0
StockOptionLevel     0
dtype: int64
Performance Missing Values
EmployeeNumber              0
PerformanceRating           0
EnvironmentSatisfaction     0
JobInvolvement              0
JobSatisfaction             0
RelationshipSatisfaction    0
WorkLifeBalance             0
dtype: int64
Experience Missing Values
EmployeeNumber             0
NumCompaniesWorked         0
TotalWorkingYears          0
TrainingTimesLastYear      0
YearsAtCompany             0
YearsInCurrentRole         0
Ye

In [32]:
employees[employees['Age'].isnull()]

,EmployeeNumber,Age,Gender,MaritalStatus,Education,EducationField,Department,JobRole,JobLevel,BusinessTravel,DistanceFromHome,EmployeeCount,Over18,StandardHours
5,8,NaN,Male,Single,2,Life Sciences,Research & Development,Laboratory Technician,1,Travel_Frequently,2,1,Y,80
27,35,NaN,Male,Married,4,Marketing,Sales,Sales Executive,2,Travel_Rarely,8,1,Y,80
134,177,NaN,Female,Married,1,Life Sciences,Human Resources,Human Resources,1,Travel_Rarely,25,1,Y,80
262,362,NaN,Male,Single,1,Technical Degree,Research & Development,Laboratory Technician,2,Travel_Rarely,2,1,Y,80
377,501,NaN,Female,Married,3,Life Sciences,Research & Development,Research Scientist,1,Travel_Rarely,2,1,Y,80


In [58]:
##Remove the 5 employees with missing 'Age' from the datasets.  5 out of 1470 employees will not caused an significant
##impact on the analysis.
employees = employees.dropna(subset=['Age'])

employees.shape


(1465, 14)

In [59]:
compensation = compensation[compensation['EmployeeNumber'].isin(employees['EmployeeNumber'])]

performance = performance[performance['EmployeeNumber'].isin(employees['EmployeeNumber'])]

experience = experience[experience['EmployeeNumber'].isin(employees['EmployeeNumber'])]

employment_status = employment_status[employment_status['EmployeeNumber'].isin(employees['EmployeeNumber'])]


In [60]:
print("Employees:", employees.shape)
print("Compensation:", compensation.shape)
print("Performance:", performance.shape)
print("Experience:", experience.shape)
print("Employment Status:", employment_status.shape)

Employees: (1465, 14)
Compensation: (1465, 7)
Performance: (1465, 7)
Experience: (1465, 8)
Employment Status: (1465, 3)


In [61]:
hr_data = pd.merge(employees, compensation, on='EmployeeNumber')

hr_data.shape


(1465, 20)

In [63]:

hr_data = pd.merge(hr_data, performance, on='EmployeeNumber')
hr_data = pd.merge(hr_data, experience, on='EmployeeNumber')
hr_data = pd.merge(hr_data, employment_status, on='EmployeeNumber')

hr_data.shape

(1465, 35)

In [64]:
# Check categorical columns for inconsistent values

for column in ['Gender', 'MaritalStatus', 'EducationField',
               'Department', 'JobRole', 'BusinessTravel',
               'Attrition', 'OverTime']:
    print(column)
    print(hr_data[column].value_counts())
    print("--------------------")


Gender
Gender
Male       875
Female     585
  Male       1
Male         1
Femal        1
M            1
m            1
Name: count, dtype: int64
--------------------
MaritalStatus
MaritalStatus
Married     670
Single      468
Divorced    327
Name: count, dtype: int64
--------------------
EducationField
EducationField
Life Sciences       603
Medical             464
Marketing           158
Technical Degree    131
Other                82
Human Resources      27
Name: count, dtype: int64
--------------------
Department
Department
Research & Development    958
Sales                     445
Human Resources            62
Name: count, dtype: int64
--------------------
JobRole
JobRole
Sales Executive              325
Research Scientist           291
Laboratory Technician        257
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               51
Name: count, dtype

In [101]:
# Remove extra spaces
hr_data['Gender'] = hr_data['Gender'].str.strip()

# Correct inconsistent entries
hr_data['Gender'] = hr_data['Gender'].replace({
    'Femal': 'Female',
    'M': 'Male',
    'm': 'Male'
})

# Check results
print(hr_data['Gender'].value_counts())

Gender
Male      879
Female    586
Name: count, dtype: int64


###**Research Question 1: Which departments and job roles have the highest attrition rates?**

####**1.1 – To identify the department with the highest employee attrition rate.**

Method: Group employees by department and calculate the percentage of employees who left.

Objective: To identify which department has the highest attrition rate and may require greater attention to employee retention.

In [66]:
hr_data.groupby('Department')['Attrition'].value_counts()

Department              Attrition
Human Resources         No            50
                        Yes           12
Research & Development  No           825
                        Yes          133
Sales                   No           353
                        Yes           92
Name: count, dtype: int64

In [100]:
# Research Question 1.1
# Calculate attrition rate by department

(hr_data.groupby('Department')['Attrition_Flag'].mean() * 100).round(2)


,Attrition_Flag
Department,
Human Resources,19.35
Research & Development,13.88
Sales,20.67


####**Findings 1.1:**

Sales recorded the highest employee attrition rate at 20.67%, followed by Human Resources at 19.35% and Research & Development at 13.88%.

Although Research & Development had the highest number of employees leaving at 133, Sales had the highest proportion of employees leaving relative to its department size.

**Conclusion:**

Sales has the highest employee attrition rate and should be prioritised for further investigation into employee retention.

####**1.2 : To identify job roles with the highest employee attrition rates by number and then by percentage.**

Method: Group employees by job role and calculate the percentage who left the company.

Objective: To identify which department has the highest employee attrition rate and determine where HR should prioritise employee retention efforts.

In [97]:
##To calculate job role attrition rate by numbers
hr_data.groupby('JobRole')['Attrition'].value_counts()


JobRole                    Attrition
Healthcare Representative  No           122
                           Yes            9
Human Resources            No            39
                           Yes           12
Laboratory Technician      No           195
                           Yes           62
Manager                    No            97
                           Yes            5
Manufacturing Director     No           135
                           Yes           10
Research Director          No            78
                           Yes            2
Research Scientist         No           244
                           Yes           47
Sales Executive            No           268
                           Yes           57
Sales Representative       No            50
                           Yes           33
Name: count, dtype: int64

In [102]:
## To calculate job role attrition rate by percentage.

hr_data['Attrition_Flag'] = (hr_data['Attrition'] == 'Yes') * 1

(hr_data.groupby('JobRole')['Attrition_Flag'].mean() * 100).sort_values(ascending=False).round(2)

,Attrition_Flag
JobRole,
Sales Representative,39.76
Laboratory Technician,24.12
Human Resources,23.53
Sales Executive,17.54
Research Scientist,16.15
Manufacturing Director,6.90
Healthcare Representative,6.87
Manager,4.90
Research Director,2.50


####**Findings 1.2**

Sales Representatives have the highest attrition rate at 39.76%, followed by Laboratory Technicians at 24.12% and Human Resources employees at 23.53%.



####**1.3: To identify which job roles has the largest number of employee departures within each department.**

Method:Filter employees with Attrition = Yes, group by Department and JobRole, and count the departures.

Objective: To identify which job roles contribute the most employee departures within each department and help HR to focus on retention strategy.

In [69]:
hr_data[hr_data['Attrition'] == 'Yes'].groupby(['Department', 'JobRole'])['Attrition'].value_counts()


Department              JobRole                    Attrition
Human Resources         Human Resources            Yes          12
Research & Development  Healthcare Representative  Yes           9
                        Laboratory Technician      Yes          62
                        Manager                    Yes           3
                        Manufacturing Director     Yes          10
                        Research Director          Yes           2
                        Research Scientist         Yes          47
Sales                   Manager                    Yes           2
                        Sales Executive            Yes          57
                        Sales Representative       Yes          33
Name: count, dtype: int64

####**Findings 1.3 :**

Human Resources: Human Resources role departure (12)
Research & Development: Laboratory Technician recorded the most departures (62).
Sales: Sales Executive recorded the most departures (57).

**Conclusion:**

Laboratory Technicians and Sales Executives contribute the largest pool of departures.

####**1.4 : Are higher attrition rates concentrated in particular job levels?**

Method: Group employees by JobLevel and calculate the attrition percentage using the mean of Attrition_Flag multiplied by 100.

Objective: To identify which job levels have the highest employee attrition rates and help HR prioritise retention efforts.



In [71]:
(hr_data.groupby('JobLevel')['Attrition_Flag'].mean() * 100).sort_values(ascending=False).round(2)

,Attrition_Flag
JobLevel,
1,26.48
3,14.68
2,9.77
5,7.25
4,4.72


####**Findings 1.4 :**

- Job Level 1: 26.48% (highest)
- Job Level 3: 14.68%
- Job Level 2: 9.77%
- Job Level 5: 7.25%
- Job Level 4: 4.72% (lowest)

**Conclusion:**

Attrition is highest at Job Level 1, suggesting that employees at this level may face greater retention challenges.
However, attrition does not decline consistently across higher job levels.

####**1.5: To examine the relationship between business travel frequency and employee attrition.**

Method: Group employees by BusinessTravel and calculate the attrition rate using the mean of Attrition_Flag multiplied by 100.

Objective: To determine whether employees who travel more frequently for business have higher attrition rates and identify potential retention concerns.



In [103]:
# Research Question 1.5
# Is the frequency of business travel associated with employee attrition?

(hr_data.groupby('BusinessTravel')['Attrition_Flag'].mean() * 100).sort_values(ascending=False).round(2)

,Attrition_Flag
BusinessTravel,
Travel_Frequently,25.00
Travel_Rarely,15.01
Non-Travel,8.00


####**Findings 1.5:**

Travel Frequently: 25.00% (highest)
Travel Rarely: 15.01%
Non-Travel: 8.00% (lowest)

**Conclusion:**

Employees who travel frequently have the highest attrition rate, approximately 3 times that of employees
who do not travel. This indicate there is an relationship between business travel frequency and employee attrition.
However, we cannot establish this is one of the reason for attrition.

HR will need to investigate the impact of frequent business travel on employee work-life balance, workload and job satisfaction.

####**1.6: To examine whether commuting distance is associated with employee attrition.**

Method: Group employees by Attrition status and calculate the average DistanceFromHome using mean().

Objective: To determine whether employees who leave tend to live farther from the workplace than those who stay nearer.  This will help to identify whether commutin g to/fro workplace is related to attrition.


In [73]:
# Research Question 1.6
# Are employees who live farther from the workplace more likely to leave?

hr_data.groupby('Attrition')['DistanceFromHome'].mean().round(2)


,DistanceFromHome
Attrition,
No,8.92
Yes,10.63


####**Findings 1.6**

Employees who stayed : Average travelling distance of 8.92.
Employees who left : Average travelling distance of 10.63.
Difference : 1.71 distance units, approximately 19.2% higher among employees who left.

**Conclusion:**

Based on the result, there could be a possible relationship between commmuting distance and employee attrition.
Further analysis is needed to determine whether attrition rates increase with distance commuting to/from workplace.

####**1.7 : Do business travel frequency and distance from home show different attrition patterns across departments and job roles?**

**Step 1: Business Travel and Attrition by Department**

Method: Group employees by Department and BusinessTravel, then calculate the attrition rate using the mean of Attrition_Flag multiplied by 100.

Objective: To identify whether business travel frequency and commuting distance are associated with different employee attrition patterns across departments and job roles, helping HR to focus on retention strategy.

In [74]:
# Research Question 1.7 Step 1
# Business travel frequency and attrition across departments

(hr_data.groupby(['Department', 'BusinessTravel'])['Attrition_Flag'].mean() * 100).round(2)


Department              BusinessTravel   
Human Resources         Non-Travel            0.00
                        Travel_Frequently    36.36
                        Travel_Rarely        17.78
Research & Development  Non-Travel            8.25
                        Travel_Frequently    20.44
                        Travel_Rarely        12.94
Sales                   Non-Travel            8.51
                        Travel_Frequently    33.33
                        Travel_Rarely        19.11
Name: Attrition_Flag, dtype: float64

####**Findings 1.7 Step 1 :**

All three departments show the same pattern; that is employees who travel frequently have the highest attrition rates.

This suggests there is a strong association between frequent business travel and attrition exists across departments.

We should also consider the number of employees in each category before a firm conclusions.

####**Step 2: Business Travel and Attrition by Job Role**

Method: Group employees by JobRole and BusinessTravel, then calculate the attrition rate using the mean of Attrition_Flag multiplied by 100.

Objective: To identify which job roles have higher attrition rates among employees who travel frequently for business and highlight potential travel-related retention concerns.

In [75]:
# Research Question 1.7 - Step 2
# Business travel frequency and attrition by job role

(hr_data.groupby(['JobRole', 'BusinessTravel'])['Attrition_Flag'].mean() * 100).round(2)


JobRole                    BusinessTravel   
Healthcare Representative  Non-Travel            6.67
                           Travel_Frequently    11.54
                           Travel_Rarely         5.56
Human Resources            Non-Travel            0.00
                           Travel_Frequently    40.00
                           Travel_Rarely        21.62
Laboratory Technician      Non-Travel            7.14
                           Travel_Frequently    32.00
                           Travel_Rarely        24.58
Manager                    Non-Travel            8.33
                           Travel_Frequently     0.00
                           Travel_Rarely         5.19
Manufacturing Director     Non-Travel            0.00
                           Travel_Frequently    10.34
                           Travel_Rarely         6.80
Research Director          Non-Travel            0.00
                           Travel_Frequently     0.00
                           Travel_Rarely         3.23
Research Scientist         Non-Travel           14.29
                           Travel_Frequently    27.78
                           Travel_Rarely        13.40
Sales Executive            Non-Travel           10.26
                           Travel_Frequently    22.03
                           Travel_Rarely        17.62
Sales Representative       Non-Travel            0.00
                           Travel_Frequently    65.22
                           Travel_Rarely        32.73
Name: Attrition_Flag, dtype: float64

####**Finding 1.7 Step 2:**

Sales Representatives who travel frequently have the highest attrition rate at 65.22%, compared with 32.73% for those who travel rarely. However, the pattern is not consistent across every job role. For example, Managers and Research Directors have 0% attrition among frequent travellers.

**Step 3: Distance from Home and Attrition by Department**

Method: Group employees by Department and Attrition status, then calculate the average DistanceFromHome using mean().

Objective: To compare the average commuting distance of employees who stayed and those who left within each department, and identify potential commuting-related retention concerns.

In [76]:
# Research Question 1.7 - Step 3
# Average distance from home by department and attrition status

hr_data.groupby(['Department', 'Attrition'])['DistanceFromHome'].mean().round(2)

Department              Attrition
Human Resources         No            7.24
                        Yes          13.42
Research & Development  No            8.99
                        Yes          10.26
Sales                   No            8.99
                        Yes          10.80
Name: DistanceFromHome, dtype: float64

####**Findings 1.7 Step 3 :**

Based on the above findings, Eemployees who left had a higher average distance from home in all three departments.
The largest difference was in Human Resources (13.42 versus 7.24).

####**Step 4: Distance from Home and Attrition by Job Role**

Method: Group employees by JobRole and Attrition status, then calculate the average DistanceFromHome using mean().

Objective: To compare the average commuting distance of employees who stayed and those who left across different job roles, and identify potential commuting-related retention concerns.

In [77]:
# Research Question 1.7 - Step 4
# Average distance from home by job role and attrition status

hr_data.groupby(['JobRole', 'Attrition'])['DistanceFromHome'].mean().round(2)

JobRole                    Attrition
Healthcare Representative  No            9.20
                           Yes          17.67
Human Resources            No            6.13
                           Yes          13.42
Laboratory Technician      No            9.41
                           Yes           9.66
Manager                    No            7.93
                           Yes          10.00
Manufacturing Director     No            9.53
                           Yes           8.80
Research Director          No            8.47
                           Yes           7.00
Research Scientist         No            8.90
                           Yes           9.77
Sales Executive            No            9.03
                           Yes          12.65
Sales Representative       No            9.00
                           Yes           8.15
Name: DistanceFromHome, dtype: float64

####**Findings Step 4 :**

Employees who left had a higher average distance from home in several job roles.

####**Conclusion:**

The relationship between distance from home and employee attrition varies across job roles. Commuting distance may be one of the contributing factor.

####**Question 1 - Overall Findings**

The analysis indicated that job role, job level, business travel and commuting distance are associated with employee attrition.

The Sales department, particularly Sales Representatives, recorded the highest attrition rates. Employees at lower job levels and those who travel frequently also showed higher attrition.

Employees who left generally lived farther from the workplace than those who stayed, although commuting patterns varied across departments and job roles.

These findings suggest that the organisation should prioritise retention investigations among Sales Representatives, lower-level employees and frequent business travellers.
However, the results show associations and do not establish that these factors directly caused employees to leave.

###**Research Question 2: How is compensation associated with employee attrition?**

####**2.1: Do employees who leave have lower average monthly income than employees who stay?**

Method:Group employees by Attrition status and calculate the average MonthlyIncome using mean().

Objective: To determine whether employees who leave earn lower average monthly income than those who stay and identify potential compensation-related retention concerns.

In [79]:
# Research Question 2.1
# Compare average monthly income between employees who stayed and left

hr_data.groupby('Attrition')['MonthlyIncome'].mean().round(2)

,MonthlyIncome
Attrition,
No,6846.27
Yes,4787.09


**Findings 2.1:**

Employees who left had a lower average monthly income (4,787.09) compared with employees who stayed (6,846.27).

The difference was 2,059.18, representing approximately 30.08% lower average monthly income.

**Conclusion:**

Lower monthly income may be associated with employee attrition. However, further analysis is needed to determine whether the difference is related to job level, job role or other factors.

####**2.2: How does monthly income vary across job levels, and is lower income associated with employee attrition?**

**Step 1: Average Monthly Income by Job Level**

Method: Group employees by JobLevel and calculate the average MonthlyIncome using mean().

Objective: To compare average monthly income across different job levels and understand how compensation varies with job seniority.

In [81]:
# Research Question 2.2 - Step 1
# Calculate average monthly income by job level

hr_data.groupby('JobLevel')['MonthlyIncome'].mean().round(2)

,MonthlyIncome
JobLevel,
1,2786.56
2,5506.04
3,9817.25
4,15503.78
5,19191.83


**Findings 2.2 Step 1:**

Average monthly income increases with job level, from 2,786.56 at Job Level 1 to 19,191.83 at Job Level 5.

**Conclusion:**

Monthly income is strongly associated with job level. Further analysis is required to determine whether employees who leave earn less than employees who stay within the same job level.

####**Step 2: Average Monthly Income by Job Level and Attrition Status**

Method: Group employees by JobLevel and Attrition status, then calculate the average MonthlyIncome using mean().

Objective: To compare the average monthly income of employees who stayed and those who left within each job level, and examine whether lower income is associated with employee attrition.

In [82]:
# Research Question 2.2 - Step 2
# Compare average monthly income by job level and attrition status

hr_data.groupby(['JobLevel', 'Attrition'])['MonthlyIncome'].mean().round(2)

JobLevel  Attrition
1         No            2854.43
          Yes           2598.13
2         No            5478.55
          Yes           5759.79
3         No            9891.03
          Yes           9388.41
4         No           15620.29
          Yes          13150.40
5         No           19170.58
          Yes          19463.80
Name: MonthlyIncome, dtype: float64

**Findings 2.2 Step 2:**

Employees who left earned lower average monthly incomes at Job Levels 1, 3 and 4.

The largest difference was at Job Level 4, where employees who left earned 2,469.89 less on average than employees who stayed.

However, employees who left at Job Levels 2 and 5 earned slightly higher average monthly incomes.

**Conclusion:**

The relationship between monthly income and attrition varies across job levels. Lower income alone does not consistently explain employee attrition.

####**2.3: Is the percentage of salary increase associated with employee attrition?**

Method: Group employees by Attrition status and calculate the average PercentSalaryHike using mean().

Objective: To compare the average salary increase percentages of employees who stayed and those who left and determine whether salary increases are associated with employee attrition.

In [83]:
# Research Question 2.3
# Compare average percentage salary increase by attrition status
hr_data.groupby('Attrition')['PercentSalaryHike'].mean().round(2)


,PercentSalaryHike
Attrition,
No,15.23
Yes,15.10


**Findings 2.3:**

Employees who stayed received an average salary increase of 15.23%, compared with 15.10% for employees who left.

The difference was only 0.13 percentage points.

**Conclusion:**

The average percentage of salary increase was almost identical between employees who stayed and those who left. This suggests that salary increase percentage is unlikely a major factor associated with employee attrition.

####**2.4: How are employees distributed across Stock Option Levels 0, 1, 2 and 3?**

Method: Use value_counts() to calculate the number of employees at each Stock Option Level.

Objective: To understand how employees are distributed across the different stock option levels and identify which levels have the highest and lowest employee representation.

In [84]:
# Research Question 2.4
# Count employees at each Stock Option Level

hr_data['StockOptionLevel'].value_counts().sort_index()


,count
StockOptionLevel,
0,629
1,593
2,158
3,85


**Findings 2.4:**

Stock Option Level 0 has the largest number of employees (629), followed by Level 1 (593), Level 2 (158) and Level 3 (85).

Approximately 83.41% of employees are concentrated in Stock Option Levels 0 and 1.

**Conclusion:**

Most employees are in the two lowest stock option levels. Further analysis is needed to examine whether stock option levels are associated with job positions, length of service and employee attrition.

####**2.5: Do stock option levels correspond to employees' job positions and length of service?**

**Step 1: Stock Option Level by Job Level**

Method: Filter employees by JobLevel, then use value_counts() to calculate the percentage of employees at each Stock Option Level within each job level.

Objective:  To examine whether employees at higher job levels tend to receive higher stock option levels and identify differences in stock option distribution across job levels.

In [87]:
# Research Question 2.5 - Step 1
# Calculate stock option percentages within each job level

for level in [1, 2, 3, 4, 5]:
    job_data = hr_data[hr_data['JobLevel'] == level]

    print("Job Level:", level)
    print((job_data['StockOptionLevel'].value_counts() / len(job_data) * 100).round(2))
    print()


Job Level: 1
StockOptionLevel
0    47.41
1    37.78
2     7.59
3     7.22
Name: count, dtype: float64

Job Level: 2
StockOptionLevel
0    40.98
1    38.72
2    15.60
3     4.70
Name: count, dtype: float64

Job Level: 3
StockOptionLevel
1    44.50
0    39.45
2    10.09
3     5.96
Name: count, dtype: float64

Job Level: 4
StockOptionLevel
1    46.23
0    40.57
2     7.55
3     5.66
Name: count, dtype: float64

Job Level: 5
StockOptionLevel
1    53.62
0    37.68
2     5.80
3     2.90
Name: count, dtype: float64



**Findings 2.5 Step 1:**

Stock Option Level 1 becomes more common at higher job levels, increasing from 37.78% at Job Level 1 to 53.62% at Job Level 5.

Stock Option Level 0 generally becomes less common at higher job levels.

However, higher job levels do not consistently correspond to higher stock option levels. For example, only 2.90% of Job Level 5 employees are at Stock Option Level 3, compared with 7.22% at Job Level 1.

**Conclusion:**

Stock option distribution varies across job levels, but higher job positions do not necessarily correspond to higher stock option levels.

**Step 2: Stock Option Level and Length of Service**

**Method:** Group employees by StockOptionLevel and calculate the average YearsAtCompany using mean().

**Objective:** To examine whether employees with longer years of service tend to have higher stock option levels.

In [88]:
# Research Question 2.5 - Step 2
# Calculate average years of service by stock option level

hr_data.groupby('StockOptionLevel')['YearsAtCompany'].mean().round(2)


,YearsAtCompany
StockOptionLevel,
0,6.58
1,7.63
2,6.79
3,6.29


**Findings 2.5 Step 2 :**

Employees at Stock Option Level 1 have the longest average service (7.63 years), while employees at Stock Option Level 3 have the shortest (6.29 years).

Average years of service do not consistently increase with higher stock option levels.

**Conclusion:**

Length of service does not show a consistent relationship with stock option levels. Other factors may influence stock option allocation.

####**2.6: Are employees with lower stock option levels more likely to leave?**

Method:  Group employees by Stock Option Level and calculate the attrition rate using the mean of Attrition_Flag multiplied by 100.

Objective: To compare employee attrition rates across different stock option levels and determine whether lower stock option levels are associated with higher employee attrition.

In [89]:
# Research Question 2.6
# Calculate attrition rate by stock option level

(hr_data.groupby('StockOptionLevel')['Attrition_Flag'].mean() * 100).round(2)

,Attrition_Flag
StockOptionLevel,
0,24.48
1,9.44
2,7.59
3,17.65


In [104]:
# Research Question 2.6 - Additional Analysis
# Examine stock option distribution by marital status

hr_data.groupby('MaritalStatus')['StockOptionLevel'].value_counts()

MaritalStatus  StockOptionLevel
Divorced       1                   195
               2                    75
               3                    49
               0                     8
Married        1                   398
               0                   153
               2                    83
               3                    36
Single         0                   468
Name: count, dtype: int64

In [105]:
# Research Question 2.6 - Additional Analysis Step 2
# Calculate attrition rate by marital status

(hr_data.groupby('MaritalStatus')['Attrition_Flag'].mean() * 100).round(2)


,Attrition_Flag
MaritalStatus,
Divorced,10.09
Married,12.54
Single,25.64


In [106]:
# Research Question 2.6 - Additional Analysis Step 3
# Calculate attrition rates by marital status and stock option level

(hr_data.groupby(['MaritalStatus', 'StockOptionLevel'])['Attrition_Flag'].mean() * 100).round(2)

MaritalStatus  StockOptionLevel
Divorced       0                   37.50
               1                    9.23
               2                    6.67
               3                   14.29
Married        0                   20.26
               1                    9.55
               2                    8.43
               3                   22.22
Single         0                   25.64
Name: Attrition_Flag, dtype: float64

**Findings 2.6:**

Employees at Stock Option Level 0 has the highest attrition rate at 24.48%, followed by Level 3 at 17.65%, Level 1 at 9.44% and Level 2 at 7.59%.

Attrition decreased from Level 0 to Level 2 before increasing again at Level 3, showing that the relationship is not consistent.

Further analysis revealed that all 468 single employees were at Stock Option Level 0. Single employees also recorded the highest attrition rate at 25.64%, compared with married at 12.54%at and divorced employees at 10.09%.

However, even among married employees, Level 0 recorded higher attrition at 20.26% than Levels 1 at 9.55% and 2 at 8.43%. A similar pattern was observed among divorced employees.

**Conclusion:**

Stock Option Level 0 is associated with higher employee attrition, but stock option levels alone do not explain the differences.

Marital status may partly influence the findings because all single employees were concentrated at Level 0. However, differences in attrition also remain within married and divorced employee groups.

HR should review stock option eligibility and retention practices while investigating other possible factors contributing to employee attrition.

###**Overall Finding for Research Question 2**

The compensation analysis indicates that employees who left had lower average monthly income than those who stayed. However, income differences varied across job levels.

Employees at Stock Option Level 0 recorded the highest attrition rate (24.48%). Further analysis showed that marital status may partly influence this relationship, as all single employees were at Level 0 and also had higher attrition rates.

Percentage salary increases showed little difference between employees who stayed and those who left.

**Overall Conclusion:**

Monthly income and stock option levels are associated with employee attrition, but compensation alone may not explain why employees leave. HR should review compensation in terms of         internal equity and external competitiveness, stock option eligibility and other potential retention factors.

###**Research Question 3: Are employees working overtime more likely to leave?**

####**3.1 What is the attrition rate for employees who work overtime compared with those who do not?**

Method: Group employees by OverTime and calculate the average Attrition_Flag multiplied by 100 to obtain the attrition rate.

Objective: To examine whether working overtime is associated with higher employee attrition.

In [90]:
# Research Question 3.1
# Calculate attrition rate by overtime status

(hr_data.groupby('OverTime')['Attrition_Flag'].mean() * 100).round(2)


,Attrition_Flag
OverTime,
No,10.49
Yes,30.53


**Findings 3.1:**

Employees who work overtime have a significantly higher attrition rate of 30.53% compared with employees who do not work overtime at 10.49%.  The difference is 20.04 percentage points.

**Conclusion:**

Working overtime is strongly associated with higher employee attrition. This suggests that HR should investigate employee workload, overtime requirements and work-life balance as potential retention concerns.


####**3.2 Does overtime-related attrition vary across departments and job roles?**

**Step 1: Overtime Attrition by Department**

Method: Group employees by Department and OverTime, then calculate the average Attrition_Flag multiplied by 100 to obtain the attrition rate.

Objective:To examine whether the relationship between overtime and employee attrition differs across departments.

In [91]:
# Research Question 3.2 - Step 1
# Calculate overtime attrition rates by department

(hr_data.groupby(['Department', 'OverTime'])['Attrition_Flag'].mean() * 100).round(2)


Department              OverTime
Human Resources         No          15.56
                        Yes         29.41
Research & Development  No           8.59
                        Yes         27.31
Sales                   No          13.88
                        Yes         37.50
Name: Attrition_Flag, dtype: float64

**Findings 3.2 – Step 1: Overtime Attrition by Department**

Employees who work overtime have higher attrition rates across all three departments.

Sales recorded the highest overtime attrition rate at 37.5%, followed by Human Resources at 29.41% and Research & Development at 27.31%.

Sales also have the largest difference in attrition rates between employees who work overtime and those who do not; a difference of 23.62 percentage points.

**Conclusion:**

Overtime is associated with higher employee attrition across all departments, particularly in Sales. This suggests that HR should investigate overtime requirements and employee workload, especially within the Sales department.

**Step 2: Overtime Attrition by Job Role**

Method: Group employees by JobRole and OverTime, then calculate the average Attrition_Flag multiplied by 100 to obtain the attrition rate.

Objective: To identify which job roles have the highest attrition rates among employees who work overtime.

In [92]:
# Research Question 3.2 - Step 2
# Calculate overtime attrition rates by job role

(hr_data.groupby(['JobRole', 'OverTime'])['Attrition_Flag'].mean() * 100).round(2)


JobRole                    OverTime
Healthcare Representative  No           7.45
                           Yes          5.41
Human Resources            No          18.42
                           Yes         38.46
Laboratory Technician      No          15.90
                           Yes         50.00
Manager                    No           1.33
                           Yes         14.81
Manufacturing Director     No           5.66
                           Yes         10.26
Research Director          No           1.75
                           Yes          4.35
Research Scientist         No           7.22
                           Yes         34.02
Sales Executive            No          11.26
                           Yes         32.98
Sales Representative       No          28.81
                           Yes         66.67
Name: Attrition_Flag, dtype: float64

**Findings 3.2 – Step 2: Overtime Attrition by Job Role**

Employees who work overtime have higher attrition rates in 8 of the 9 job roles.

Sales Representatives recorded the highest overtime attrition rate at 66.67%, followed by Laboratory Technicians at 50.00% and Human Resources employees at 38.46%.

Healthcare Representatives were the only exception, with a slightly lower attrition rate among employees working overtime at 5.41% compared with those who did not at 7.45%.

**Conclusion:**

Overtime is associated with higher employee attrition in most job roles, particularly Sales Representatives and Laboratory Technicians.

####**3.3 Is work-life balance associated with attrition among employees working overtime?**

Method: Filter employees whose OverTime status is Yes, group them by WorkLifeBalance and calculate the average Attrition_Flag multiplied by 100 to obtain the attrition rate.

Objective: To examine whether employees working overtime with lower work-life balance ratings have higher attrition rates.



In [93]:
# Research Question 3.3
# Calculate attrition rates by work-life balance
# for employees working overtime

overtime_data = hr_data[hr_data['OverTime'] == 'Yes']

(overtime_data.groupby('WorkLifeBalance')['Attrition_Flag'].mean() * 100).round(2)

,Attrition_Flag
WorkLifeBalance,
1,45.45
2,30.77
3,28.74
4,33.33


In [99]:
# Research Question 3.3 - Additional Analysis
# Count employees who stayed and left by work-life balance

overtime_data.groupby('WorkLifeBalance')['Attrition'].value_counts()

WorkLifeBalance  Attrition
1                No            12
                 Yes           10
2                No            72
                 Yes           32
3                No           181
                 Yes           73
4                No            24
                 Yes           12
Name: count, dtype: int64

**Findings: 3.3**

Among employees working overtime, those with the poorest work-life balance (Rating 1) has the highest attrition rate of 45.45%. However, this was based on only 22 employees, of whom 10 left.

Attrition decreased to 30.77% at Rating 2 and 28.74% at Rating 3, before increasing to 33.33% at Rating 4.

Rating 3 had the largest employee group of 254 employees and the lowest attrition rate of 28.74%. Rating 4 had only 36 employees.

**Conclusion:**

Poor work-life balance appears to be associated with higher employee attrition among employees working overtime. However, the relationship is not consistent across all ratings, and the smaller employee groups at Ratings 1 and 4 mean that the results should be interpreted cautiously.

HR should investigate workload, overtime requirements and work-life balance, particularly among employees reporting poor work-life balance.

